# Lab 2: Grouping & Aggregating Data

## Objectives

By the end of this lab, you should be able to:

- Group a dataset by one or more variables
- Compute summary statistics with `.agg()`
- Count observations several different ways
- Extract specific rows from within each group
- Combine grouping with filtering to answer real analysis questions

## Setup

We'll use two datasets you've seen before: `nycflights13`, and `fastfood`: a nutrition dataset covering menu items from eight fast food restaurants.

In [1]:
!pip install nycflights13 pandas

import pandas as pd
import nycflights13 as nf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 63.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nycflights13: filename=nycflights13-0.0.3-py3-none-any.whl size=8732782 sha256=bfbb3befb80b1082dcbec97ca1b8608c9fcc9b2b75f870e91a5091a0d6cff5f4
  Stored in directory: /Users/xpz5km/Library/Caches/pip/wheels/64/33/b8/55dbea9d199e6f4869488e0b029d09d91c34c4f4706f232253
Successfully built nycflights13


/opt/anaconda3/lib/python3.13/site-packages/nycflights13/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename as _rf


## 1. First look at the data

We begin by loading the `fastfood` dataset and examining the first few rows to understand its structure. A CSV copy is included alongside this notebook.

In [43]:
# load the fastfood dataset from csv file

## 2. Grouping basics

Use `.groupby()` to divide your dataset into groups meaningful for your analysis. Let's group the `flights` dataset by month.

In [44]:
#group flights by month
flights = nf.flights

Notice that just calling `.groupby()` on its own doesn't show you any data. Instead, it produces a `GroupBy` object, an intermediate structure waiting for you to tell it what to compute. It only produces an actual DataFrame once you follow it with some other operation such as aggregating values using `.agg()`.

## 3. Aggregating with `.agg()`

The most important grouped operation is a summary. `.agg()` (short for aggregate) computes one or more statistics per group. The following computes the average departure delay by month:

In [45]:
# build on groupby to calculate average departure delay for each month

Note: `"mean"` (and most other aggregations) ignore missing values by default. You don't need to do anything special to handle `NaN` values here.

We can do the same thing with our fastfood data. Let's find the average calories per restaurant:

In [46]:
# find the average calories per restaurant in the fastfood dataset

## 4. Counting observations

`.value_counts()` is a quick way to count how many rows fall into each category of a single column:

In [47]:
# find the number of entries for each restaurant in the fastfood dataset

By default this names the result after the column it counted, with values as `count`. To get a labeled DataFrame with a custom count-column name, convert it and rename:

In [48]:
# rename the counts to total_items

## 5. Group size inside `.agg()`

Another way to count observations per group is to include `"size"` inside `.agg()`. `.value_counts()` is essentially a shortcut for exactly this pattern:

In [49]:
# perform the sme operation but using the agg method instead of value_counts

This is especially useful because it lets you compute a count *alongside* other summary statistics in the same call.

## 6. Multiple summary statistics in one call

We can compute several summaries at once. Below, we find the number of items, and the average calories and sugar, per restaurant.

In [50]:
# compute the number of items, average calories, and average sugar per restaurant in one go.

## 7. Filtering an aggregated result

Sometimes we want to analyze specific aspects of the summarized data. Here, we calculate average sodium per restaurant and then filter down to only restaurants with an average sodium level above 1000mg. Aggregate first, then filter the *aggregated* result. `.query()` operates on whatever DataFrame comes right before it in the chain:

In [51]:
# find all restaurants with an average sodium content greater than 1000 mg per item


## 8. Extracting specific rows per group

Sometimes you don't just want a summary statistic. You may want the *entire row* that produced it. For example, the following code finds the flights that are most delayed upon arrival at each destination: sort within each group descending, then take the top row(s) of each group.

In [52]:

# find the flight with the longest arrival delay for each destination, and return all columns with destination first, followed by the other columns in their original order


This gives you the whole corresponding row instead of just the summary statistic. (Note: unlike some other tools, `.head(1)` here does not automatically keep tied values. if you need ties included, you'd instead filter each group to rows equal to the group's max.)

## 9. Grouping by multiple variables

You can group by more than one variable at once. For example, we could make a group for each date.

In [53]:
# find the number of flights for each day in the flights dataset as a new column called n


We can get more granular information by grouping by `month` and `origin` together:

In [54]:
# find the average departure delay and number of flights for each month and origin in the flights dataset


Or break it down further, by airline as well:

In [55]:
# find the average departure delay and number of flights for each month, origin, and carrier in the flights dataset. Query the grouped data for flights originating from EWR in January 
(
    flights
    .groupby(["month", "origin", "carrier"])
    .agg(
        avg_dep_delay=("dep_delay", "mean"),
        num_flights=("dep_delay", "size")
    )
    .reset_index()
    .query('origin == "EWR" and month == 1')
)

,month,origin,carrier,avg_dep_delay,num_flights
0,1,EWR,9E,12.870130,82
1,1,EWR,AA,10.937500,298
2,1,EWR,AS,7.354839,62
3,1,EWR,B6,10.947276,573
4,1,EWR,DL,6.919118,279
5,1,EWR,EV,24.888041,3838
6,1,EWR,MQ,13.313725,212
7,1,EWR,UA,8.675193,3657
8,1,EWR,US,1.453521,363
9,1,EWR,WN,9.727447,529


## 10. Treating the whole DataFrame as one group

A `.groupby()` object is a temporary, lazy wrapper. This means that it doesn't permanently change the underlying DataFrame. If you want a single, overall summary rather than a per-group one, just call `.agg()` directly on the whole DataFrame:

In [56]:
flights.agg(
    avg_delay=("dep_delay", "mean"),
    flights=("dep_delay", "size")
)

,dep_delay
avg_delay,12.63907
flights,336776.00000


This treats the entire DataFrame as a single group, giving you one summary row for the whole dataset.

## 11. Reshaping long datasets with .pivot_table()

Everything so far in this lab produces a long table: one row per group. That's the right shape for most analysis, but sometimes you want a table shaped for a human to scan, with one category running across the columns instead of down the rows, so you can compare values side by side.

.pivot_table() does this in one step: it groups, aggregates, and reshapes all at once. Compare this to the month/origin summary from Section 9: same underlying computation, but here origin becomes columns instead of staying as a row value.

In [ ]:
# pivot the flights dataset to show the average departure delay for each month and origin in a table format, with months as rows and origins as columns

Notice this is the same information as flights.groupby(["month", "origin"]).agg(avg_dep_delay=("dep_delay", "mean")), just reshaped. .pivot_table() is the tool to reach for when the output needs to be wide. When you need to go the other direction (wide back to long), that's a different tool called .melt(), which we'll use in Lab 4 once we need long-format data for plotting.

## On your own

In this challenge, you'll apply everything from this lab to investigate patterns in the `fastfood` dataset. Build a small summary table comparing menu items across restaurants using filtering, a new column, grouping, and aggregation.

**Task 1.** Create a new DataFrame that includes only the restaurants `"Mcdonalds"`, `"Burger King"`, and `"Subway"`, with menu items where `calories > 300`.

**Task 2.** Add a new column called `calories_from_fat_pct`, computed as `(cal_fat / calories) * 100`.

**Task 3.** Group the data by restaurant and compute: average calories, average total fat, average calories-from-fat percentage, and number of menu items.

**Task 4.** Sort your final table from highest to lowest average calories, and print it.

Notice this is the same information as flights.groupby(["month", "origin"]).agg(avg_dep_delay=("dep_delay", "mean")), just reshaped. .pivot_table() is the tool to reach for when the output needs to be wide. When you need to go the other direction (wide back to long), that's a different tool called .melt(), which we'll use in Lab 4 once we need long-format data for plotting.

In [ ]:
# Complete Tasks 1-4

**Task 5**

pd.cut() divides a numeric column into bins and labels each row with the bin it falls into, similar to how you might describe a menu item as "low," "medium," or "high" calorie rather than giving its exact calorie count. We haven't covered it yet, but here's the pattern you need:

`df["new_column"] = pd.cut(df["numeric_column"], bins=3, labels=["Low", "Medium", "High"])`

Using three_fastfood_subset from Task 1, create a new column called calorie_tier by applying pd.cut() to calories, with bins=3 and labels=["Low", "Medium", "High"]. Then build a pivot_table showing average sodium, with restaurant as rows and your new calorie_tier as columns. Print the result below.

In [ ]:
# Task 5: derive calorie_tier, then pivot wide by restaurant and calorie_tier

## Submission

Export this notebook to HTML (`Jupyter: Export to HTML` from the Command Palette, or the "..." menu in the notebook toolbar) and submit the HTML file on Canvas.